# 5. Pose Quality Hard Case Exploration

## Purpose

This notebook identifies hard cases in the LLSP dataset using **pose extraction quality alone** — no model predictions required.

The goal is to close with evidence about what makes certain videos structurally difficult:
- Which exercises have the worst pose detection quality?
- Which keypoints are most often missing or low-confidence by exercise type?
- Which specific videos have visibility or detection problems?

This evidence supports the decision point: *is the data quality sufficient to build a reliable rep-counting product on this dataset?*

---

## Notebook Structure

1. Setup & Data Loading
2. Extraction Quality Overview (from `pose_extraction_report.csv`)
3. Per-Exercise Quality Breakdown
4. Keypoint Confidence Analysis (from `.npy` files — requires Drive/Colab)
5. Hard Case Identification
6. Evidence Summary

**Sections 1–3** can run locally from the tracked CSVs.  
**Section 4** requires the `.npy` pose files and is designed to run on Google Colab with Drive mounted.

## 1. Setup & Data Loading

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Libraries loaded')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
# Detect whether running locally or on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
    POSE_FEATURE_DIR = DRIVE_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned' / 'pose_features'
except ImportError:
    IN_COLAB = False
    REPO_ROOT = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path('..') / '..'
    POSE_FEATURE_DIR = REPO_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned' / 'pose_features'

# These CSVs are tracked in the repo — always available
NOTEBOOK_DIR = Path('.').resolve()
REPO_ROOT_LOCAL = NOTEBOOK_DIR.parents[1]
META_DIR = REPO_ROOT_LOCAL / 'datasets' / 'metadata' / 'llsp'

REPORT_CSV           = META_DIR / 'pose_extraction_report.csv'
REPORT_REMAINING_CSV = META_DIR / 'pose_extraction_report_remaining.csv'
INDEX_CSV            = META_DIR / 'pose_feature_index.csv'
TRAIN_CSV            = META_DIR / 'train.csv'
VALID_CSV            = META_DIR / 'valid.csv'

print(f'IN_COLAB                    : {IN_COLAB}')
print(f'REPORT_CSV exists           : {REPORT_CSV.exists()}')
print(f'REPORT_REMAINING_CSV exists : {REPORT_REMAINING_CSV.exists()}')
print(f'INDEX_CSV  exists           : {INDEX_CSV.exists()}')
print(f'TRAIN_CSV  exists           : {TRAIN_CSV.exists()}')
print(f'POSE_FEATURE_DIR            : {POSE_FEATURE_DIR}')
print(f'  exists                    : {POSE_FEATURE_DIR.exists()}')

In [ ]:
# ── Load tracked CSVs ─────────────────────────────────────────────────────
report_parts = [pd.read_csv(REPORT_CSV)]
if REPORT_REMAINING_CSV.exists():
    report_parts.append(pd.read_csv(REPORT_REMAINING_CSV))
report_df = pd.concat(report_parts, ignore_index=True)

index_df = pd.read_csv(INDEX_CSV)
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

# Normalize video name across all sources
for df in [report_df, index_df, train_df, valid_df]:
    df['name'] = df['name'].astype(str).str.strip()

# Join index (has exercise type + split) onto report (has extraction status)
merged = report_df.merge(
    index_df[['name', 'type', 'split', 'count']].drop_duplicates('name'),
    on='name',
    how='left'
)

print(f'Report rows  : {len(report_df)}  (initial: {len(report_parts[0])}, remaining: {len(report_parts[1]) if len(report_parts) > 1 else 0})')
print(f'Index rows   : {len(index_df)}')
print(f'Merged rows  : {len(merged)}')
print(f'\nStatus counts:')
print(merged['status'].value_counts().to_string())
print(f'\nExercise types in index:')
print(index_df['type'].value_counts().to_string())

## 2. Extraction Quality Overview

### Interpretation
- `ok` — pose features written successfully
- `ok_with_zero_pose_frames` — YOLO ran but detected no person in any frame
- `failed` — video not found or extraction error
- `skipped_exists` — file already existed from a previous run

Any non-`ok` status represents a video where we have **no usable pose signal**.

In [ ]:
total = len(merged)
status_counts = merged['status'].value_counts()
ok_count = status_counts.get('ok', 0) + status_counts.get('skipped_exists', 0)
problem_count = total - ok_count

print('=' * 50)
print('POSE EXTRACTION QUALITY OVERVIEW')
print('=' * 50)
print(f'Total videos processed : {total}')
print(f'Successfully extracted : {ok_count}  ({100*ok_count/total:.1f}%)')
print(f'Problems               : {problem_count}  ({100*problem_count/total:.1f}%)')
print()
print('Status breakdown:')
for status, cnt in status_counts.items():
    print(f'  {status:<30} {cnt:>4}  ({100*cnt/total:.1f}%)')

# Frame coverage — fraction of frames where YOLO tracked a person
ok_df = merged[merged['status'].isin(['ok', 'skipped_exists'])].copy()
ok_df['frames_total'] = pd.to_numeric(ok_df['frames_total'], errors='coerce')
ok_df['frames_used']  = pd.to_numeric(ok_df['frames_used'],  errors='coerce')
ok_df['frame_coverage'] = ok_df['frames_used'] / ok_df['frames_total'].replace(0, np.nan)

print(f'\nFrame coverage (ok videos only):')
print(f'  Mean  : {ok_df["frame_coverage"].mean():.3f}')
print(f'  Median: {ok_df["frame_coverage"].median():.3f}')
print(f'  Min   : {ok_df["frame_coverage"].min():.3f}')

low_coverage  = ok_df[ok_df['frame_coverage'] < 0.95]
very_low      = ok_df[ok_df['frame_coverage'] < 0.50]
print(f'  Videos with <95% frame coverage : {len(low_coverage)}  ({100*len(low_coverage)/len(ok_df):.1f}%)')
print(f'  Videos with <50% frame coverage : {len(very_low)}   ({100*len(very_low)/len(ok_df):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status distribution
colors = ['#4caf50' if s in ('ok', 'skipped_exists') else '#f44336'
          for s in status_counts.index]
axes[0].barh(status_counts.index, status_counts.values, color=colors, edgecolor='white')
axes[0].set_xlabel('Number of videos')
axes[0].set_title('Pose Extraction Status', fontweight='bold')
for i, v in enumerate(status_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=10)

# Frame count distribution (ok only)
axes[1].hist(ok_df['frames_total'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(ok_df['frames_total'].median(), color='red', linestyle='--',
                label=f'Median = {ok_df["frames_total"].median():.0f}')
axes[1].set_xlabel('Total Frames per Video')
axes[1].set_ylabel('Count')
axes[1].set_title('Frame Count Distribution (Successful Extractions)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('pose_extraction_overview.png', bbox_inches='tight')
plt.show()

## 3. Per-Exercise Quality Breakdown

### Interpretation
- A high failure rate for a specific exercise indicates a structural visibility problem with that exercise type (e.g., ground-level exercises like push-ups, or highly dynamic exercises like jump jacks).
- Even among `ok` extractions, low mean frame counts may mean YOLO only tracked the person for part of the video.

In [ ]:
merged['frames_total'] = pd.to_numeric(merged['frames_total'], errors='coerce')
merged['frames_used']  = pd.to_numeric(merged['frames_used'],  errors='coerce')
merged['frame_coverage'] = merged['frames_used'] / merged['frames_total'].replace(0, np.nan)

exercise_stats = (
    merged.groupby('type')
    .agg(
        total=('name', 'count'),
        ok=('status', lambda s: s.isin(['ok', 'skipped_exists']).sum()),
        failed=('status', lambda s: (s == 'failed').sum()),
        zero_pose=('status', lambda s: (s == 'ok_with_zero_pose_frames').sum()),
        mean_frames=('frames_total', 'mean'),
        mean_coverage=('frame_coverage', 'mean'),
    )
    .reset_index()
)
exercise_stats['success_rate'] = (exercise_stats['ok'] / exercise_stats['total'] * 100).round(1)
exercise_stats['problem_rate'] = (100 - exercise_stats['success_rate']).round(1)
exercise_stats = exercise_stats.sort_values('mean_coverage')  # worst tracking coverage first

print('Per-Exercise Extraction Quality (sorted by frame coverage, worst first):')
print(exercise_stats[['type', 'total', 'ok', 'failed', 'zero_pose',
                       'success_rate', 'mean_coverage', 'mean_frames']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Frame coverage by exercise (the meaningful quality signal when all extractions succeed)
coverage_pct = exercise_stats['mean_coverage'] * 100
colors = ['#f44336' if c < 95 else '#4caf50' for c in coverage_pct]
bars = axes[0].barh(exercise_stats['type'], coverage_pct,
                    color=colors, edgecolor='white')
axes[0].axvline(95, color='orange', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Mean Frame Coverage (%)')
axes[0].set_title('YOLO Person Tracking Coverage by Exercise\n(% of frames where person was tracked)', fontweight='bold')
axes[0].set_xlim(0, 105)
axes[0].legend()
for bar, val in zip(bars, coverage_pct):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# Mean frame count by exercise
axes[1].barh(exercise_stats['type'], exercise_stats['mean_frames'],
             color='steelblue', edgecolor='white')
axes[1].set_xlabel('Mean Frame Count')
axes[1].set_title('Mean Video Length by Exercise (frames)', fontweight='bold')

plt.suptitle('Per-Exercise Pose Extraction Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pose_quality_by_exercise.png', bbox_inches='tight')
plt.show()

## 4. Keypoint Confidence Analysis

**Requires Google Colab with Drive mounted** — loads `.npy` pose feature files.

Each `.npy` file has shape `[T, 51]` where T = number of frames and 51 = 17 keypoints × 3 values (x, y, confidence).

YOLO keypoint order (0-indexed):
`nose, left_eye, right_eye, left_ear, right_ear, left_shoulder, right_shoulder,`  
`left_elbow, right_elbow, left_wrist, right_wrist, left_hip, right_hip,`  
`left_knee, right_knee, left_ankle, right_ankle`

### Interpretation
- Mean confidence < 0.5 for a keypoint means YOLO frequently cannot locate it in that exercise's videos.
- Exercises with many low-confidence keypoints are structurally harder for pose-based counting.
- This tells us whether the problem is the data or the model.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
else:
    print('Running locally — skipping Drive mount.')
    print(f'POSE_FEATURE_DIR: {POSE_FEATURE_DIR}')
    print(f'  exists: {POSE_FEATURE_DIR.exists()}')

In [ ]:
KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]
N_KEYPOINTS = len(KEYPOINT_NAMES)
CONF_THRESHOLD = 0.5  # YOLO confidence threshold for "visible"

print(f'Keypoints: {N_KEYPOINTS}')
print(f'Confidence threshold for "visible": {CONF_THRESHOLD}')

In [ ]:
def load_pose_features(row):
    """Load a single .npy pose file. Returns None if unavailable."""
    # Try repo-relative path first, then Drive path
    candidates = [
        POSE_FEATURE_DIR / row['name'].replace('.mp4', '.npy'),
    ]
    if pd.notna(row.get('feature_path', None)):
        candidates.append(Path(str(row['feature_path'])))

    for path in candidates:
        if path.exists():
            arr = np.load(str(path))
            if arr.ndim == 2 and arr.shape[1] == 51:
                return arr
    return None


def compute_confidence_stats(pose_array):
    """Extract per-keypoint confidence statistics from a [T, 51] array."""
    conf_indices = [k * 3 + 2 for k in range(N_KEYPOINTS)]
    conf = pose_array[:, conf_indices]  # [T, 17]
    return {
        'mean_conf': conf.mean(axis=0),           # [17] mean conf per keypoint
        'visible_rate': (conf >= CONF_THRESHOLD).mean(axis=0),  # [17] fraction of frames visible
        'n_frames': len(pose_array),
    }


# Check availability before loading all files
sample_row = ok_df.iloc[0]
sample_arr = load_pose_features(sample_row)
if sample_arr is not None:
    print(f'Sample video: {sample_row["name"]}')
    print(f'  Array shape: {sample_arr.shape}  (frames × 51)')
    print(f'  Mean confidence first 3 keypoints: {sample_arr[:, [2, 5, 8]].mean(axis=0).round(3)}')
    FEATURES_AVAILABLE = True
else:
    print('Pose feature files not available in this environment.')
    print('Run on Colab with Drive mounted to load .npy files.')
    FEATURES_AVAILABLE = False

In [ ]:
if not FEATURES_AVAILABLE:
    print('Skipping .npy analysis — feature files not available.')
else:
    records = []
    failed_load = []

    for _, row in ok_df.iterrows():
        arr = load_pose_features(row)
        if arr is None:
            failed_load.append(row['name'])
            continue
        stats = compute_confidence_stats(arr)
        record = {
            'name': row['name'],
            'type': row.get('type', 'unknown'),
            'split': row.get('split', 'unknown'),
            'n_frames': stats['n_frames'],
        }
        for k, kname in enumerate(KEYPOINT_NAMES):
            record[f'mean_conf_{kname}'] = stats['mean_conf'][k]
            record[f'visible_rate_{kname}'] = stats['visible_rate'][k]
        record['overall_mean_conf'] = stats['mean_conf'].mean()
        record['overall_visible_rate'] = stats['visible_rate'].mean()
        records.append(record)

    conf_df = pd.DataFrame(records)
    print(f'Loaded: {len(conf_df)} videos  |  Failed to load: {len(failed_load)}')
    if failed_load:
        print(f'Failed: {failed_load[:10]}')
    print(f'\nOverall confidence stats:')
    print(conf_df['overall_mean_conf'].describe().round(3).to_string())

In [ ]:
if not FEATURES_AVAILABLE:
    print('Skipping keypoint heatmap — feature files not available.')
else:
    # Mean visible rate per keypoint per exercise
    visible_cols = [f'visible_rate_{k}' for k in KEYPOINT_NAMES]
    heatmap_data = (
        conf_df.groupby('type')[visible_cols]
        .mean()
        .rename(columns={f'visible_rate_{k}': k for k in KEYPOINT_NAMES})
    )

    fig, ax = plt.subplots(figsize=(18, max(6, len(heatmap_data) * 0.6)))
    sns.heatmap(
        heatmap_data,
        annot=True, fmt='.2f',
        cmap='RdYlGn',
        vmin=0, vmax=1,
        linewidths=0.4,
        ax=ax,
        cbar_kws={'label': f'Fraction of frames with conf ≥ {CONF_THRESHOLD}'}
    )
    ax.set_title(
        f'Keypoint Visibility Rate by Exercise\n'
        f'(fraction of frames where confidence ≥ {CONF_THRESHOLD})',
        fontweight='bold'
    )
    ax.set_xlabel('Keypoint')
    ax.set_ylabel('Exercise')
    plt.xticks(rotation=40, ha='right')
    plt.tight_layout()
    plt.savefig('keypoint_visibility_heatmap.png', bbox_inches='tight')
    plt.show()

    # Flag keypoints with <50% visibility in any exercise
    print('\nKeypoints with mean visibility < 50% for at least one exercise:')
    for kname in KEYPOINT_NAMES:
        col = f'visible_rate_{kname}'
        low = conf_df.groupby('type')[col].mean()
        worst = low[low < 0.5]
        if not worst.empty:
            print(f'  {kname:<20}: ' + ', '.join(f'{ex}={v:.2f}' for ex, v in worst.items()))

In [ ]:
if not FEATURES_AVAILABLE:
    print('Skipping worst-video analysis — feature files not available.')
else:
    # Worst videos per exercise by overall visibility
    print('Bottom 5 videos by overall keypoint visibility rate per exercise:\n')
    for exercise in sorted(conf_df['type'].dropna().unique()):
        sub = conf_df[conf_df['type'] == exercise].nsmallest(5, 'overall_visible_rate')
        if sub.empty:
            continue
        print(f'{exercise}:')
        print(
            sub[['name', 'n_frames', 'overall_mean_conf', 'overall_visible_rate']]
            .round(3)
            .to_string(index=False)
        )
        print()

## 5. Hard Case Identification

Hard cases are defined purely from pose signal quality — no model predictions needed:
- `pose_failure`: extraction status not `ok`
- `low_overall_visibility`: overall visible rate below threshold
- `critical_keypoint_missing`: exercise-critical keypoints frequently absent

### Critical keypoints by exercise
| Exercise | Critical keypoints |
|---|---|
| squat | hip, knee, ankle |
| pull_up | shoulder, elbow, wrist |
| push_up | shoulder, elbow, wrist, hip |
| sit_up | shoulder, hip, knee |
| jump_jacks | shoulder, hip, ankle |
| bench_pressing | shoulder, elbow, wrist |
| front_raise | shoulder, elbow, wrist |
| battle_rope | shoulder, elbow, wrist |
| pommelhorse | shoulder, elbow, wrist, hip |

In [ ]:
CRITICAL_KEYPOINTS = {
    'squat':          ['left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle'],
    'pull_up':        ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'],
    'push_up':        ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip'],
    'sit_up':         ['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip', 'left_knee', 'right_knee'],
    'jump_jacks':     ['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip', 'left_ankle', 'right_ankle'],
    'bench_pressing': ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'],
    'front_raise':    ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'],
    'battle_rope':    ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'],
    'pommelhorse':    ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip'],
}
VISIBILITY_THRESHOLD    = 0.40  # overall keypoint visible rate below this → hard case
CRITICAL_KP_THRESHOLD   = 0.30  # critical keypoint visibility below this → critical missing
COVERAGE_THRESHOLD      = 0.50  # frame coverage below this → tracking failure

# ── Tag from extraction report (always available) ──
failed_videos = set(merged[~merged['status'].isin(['ok', 'skipped_exists'])]['name'])
low_cov_videos = set(ok_df[ok_df['frame_coverage'] < COVERAGE_THRESHOLD]['name'])
print(f'Videos with extraction failure    : {len(failed_videos)}')
print(f'Videos with <{int(COVERAGE_THRESHOLD*100)}% frame coverage : {len(low_cov_videos)}')

if not FEATURES_AVAILABLE:
    print('\nConfidence-based tagging requires .npy files. Showing coverage-based hard cases.')
    hard_cases = ok_df[ok_df['frame_coverage'] < COVERAGE_THRESHOLD][['name', 'type', 'split', 'status', 'frame_coverage']].copy()
    hard_cases['hard_case_reason'] = 'low_frame_coverage'
    hard_cases = hard_cases.sort_values('frame_coverage')
    print(f'\nHard cases (low frame coverage): {len(hard_cases)}')
    print(hard_cases.to_string(index=False))
else:
    def tag_hard_case(row):
        reasons = []
        if row['name'] in failed_videos:
            reasons.append('pose_failure')
        if row['name'] in low_cov_videos:
            reasons.append('low_frame_coverage')
        if row.get('overall_visible_rate', 1.0) < VISIBILITY_THRESHOLD:
            reasons.append('low_overall_visibility')
        exercise = row.get('type')
        if exercise in CRITICAL_KEYPOINTS:
            for kp in CRITICAL_KEYPOINTS[exercise]:
                col = f'visible_rate_{kp}'
                if col in row and row[col] < CRITICAL_KP_THRESHOLD:
                    reasons.append(f'critical_kp_missing:{kp}')
                    break
        return '|'.join(reasons) if reasons else 'ok'

    conf_df['hard_case_reason'] = conf_df.apply(tag_hard_case, axis=1)
    hard_cases = conf_df[conf_df['hard_case_reason'] != 'ok'].copy()

    print(f'\nHard cases identified: {len(hard_cases)} / {len(conf_df)} ({100*len(hard_cases)/len(conf_df):.1f}%)')
    print('\nHard case count by exercise:')
    print(hard_cases.groupby('type').size().sort_values(ascending=False).to_string())

    print('\nHard case reason breakdown:')
    from collections import Counter
    reasons_flat = [r for reasons in hard_cases['hard_case_reason'] for r in reasons.split('|')]
    for reason, cnt in Counter(reasons_flat).most_common():
        print(f'  {reason:<40} {cnt}')

## 6. Evidence Summary

### Interpretation guide

Fill this section after running the analysis. The questions below structure the conclusions:

1. **What fraction of videos have unusable pose signal?**  
   → Look at extraction success rate overall and by exercise.

2. **Which exercises have the worst keypoint visibility?**  
   → Look at the heatmap. Exercises with dark red cells in critical keypoints are structurally hard.

3. **Is the problem concentrated in specific exercises or systemic?**  
   → If only 1–2 exercises are problematic, dropping them might be viable.  
   → If the problem is across most exercises, the dataset is not usable as-is.

4. **Is the failure mode something a better dataset would fix?**  
   → Occlusion / side-angle / gym equipment blocking keypoints is a data collection problem.  
   → If YouTube-style videos systematically have these issues, we need a curated dataset instead.

In [ ]:
print('=' * 60)
print('HARD CASE EVIDENCE SUMMARY')
print('=' * 60)

print(f'\n1. Extraction quality')
print(f'   Total videos     : {total}')
print(f'   Successful       : {ok_count}  ({100*ok_count/total:.1f}%)')
print(f'   Failed / no pose : {problem_count}  ({100*problem_count/total:.1f}%)')

print(f'\n2. Worst exercises by extraction success rate:')
worst_ex = exercise_stats.nsmallest(3, 'success_rate')[['type', 'total', 'ok', 'success_rate']]
print(worst_ex.to_string(index=False))

if FEATURES_AVAILABLE:
    print(f'\n3. Overall keypoint visibility (mean across all videos):')
    overall_vis = conf_df[[f'visible_rate_{k}' for k in KEYPOINT_NAMES]].mean()
    overall_vis.index = [i.replace('visible_rate_', '') for i in overall_vis.index]
    low_kps = overall_vis[overall_vis < 0.7]
    if not low_kps.empty:
        print('   Keypoints with <70% overall visibility:')
        for kp, val in low_kps.sort_values().items():
            print(f'     {kp:<25} {val:.2f}')
    else:
        print('   All keypoints have ≥70% overall visibility.')

    print(f'\n4. Hard cases: {len(hard_cases)} videos ({100*len(hard_cases)/len(conf_df):.1f}%)')
    print(f'   By exercise:')
    print(hard_cases.groupby('type').size().sort_values(ascending=False).to_string())

print('\n' + '=' * 60)
print('CONCLUSION (fill after review):')
print('-' * 60)
print('[ ] Data quality is sufficient for the product goal.')
print('[x] Data quality is NOT sufficient — specific issues found above.')
print('    → Decision: find a new dataset with controlled recording conditions.')
print('=' * 60)